# Atividade Prática — Limpeza de Dados

**Módulo/Bloco 1 — Semana 6 · Visualização de Dados e Business Intelligence**

Nesta atividade você vai aplicar, em uma base nova, tudo o que vimos na aula:
diagnóstico geral, tratamento de dados ausentes, remoção de duplicatas, detecção de
outliers e normalização.

**Como usar este notebook:**
- Cada exercício tem uma célula de markdown com o enunciado e, logo depois, uma célula
  de código vazia (com um comentário `# Seu código aqui`) para você escrever a resposta.
- Pode criar quantas células extras precisar.
- Não existe uma única resposta "certa" para todos os exercícios — o importante é
  justificar a escolha (ex: por que usar mediana em vez de média).

Rode a célula abaixo primeiro para carregar as bibliotecas e a base de dados.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
%matplotlib inline
sns.set_style('whitegrid')


## A base de dados

Uma base fictícia de clientes de um e-commerce, com problemas de limpeza inseridos de
propósito (ausências, duplicatas, outliers e inconsistências de formato) — assim como
na base usada em aula, mas com valores diferentes.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

from google.colab import userdata
DATABASE_URL = userdata.get('DB_URL')

# Carregue a variavel DATABASE_URL conforme o seu ambiente (Colab ou VS Code)
engine = create_engine(DATABASE_URL)

# Leitura da tabela principal
df_vendas = pd.read_sql("SELECT * FROM clientes_sujos;", engine)

query = "SELECT * FROM clientes_sujos where estado = 'MG'"
df2 =  pd.read_sql(query, engine)
# df_vendas.head() # se for no VS Code, use print(df_vendas.head())
# df_vendas.columns  # se for no VS Code, use print(df_vendas.columns)

df_vendas

,id,id_cliente,nome,cpf,cidade,estado,idade,data_cadastro,categoria_favorita,valor_gasto
0,1,1,Beatriz Nogueira,101.101.101-01,Curitiba,PR,29.0,2023-03-10,Livros,450.0
1,2,2,Otávio Ramos,202.202.202-02,curitiba,pr,33.0,11/03/2023,Games,890.0
2,3,3,Renata Alves,303.303.303-03,Belo Horizonte,MG,NaN,2023-03-14,Beleza,120.0
3,4,4,Lucas Farias,404.404.404-04,None,SP,41.0,2023-03-15,Games,NaN
4,5,5,Otávio Ramos,202.202.202-02,curitiba,pr,33.0,11/03/2023,Games,890.0
5,6,5,Otávio Ramos,202.202.202-02,curitiba,pr,33.0,11/03/2023,Games,890.0
6,7,6,Camila Duarte,505.505.505-05,Recife,PE,37.0,2023-03-18,Livros,210.0
7,8,7,Vitor Sales,606.606.606-06,-,sp,210.0,2023-03-19,Beleza,75.5
8,9,8,Priscila Matos,707.707.707-07,Salvador,BA,26.0,20/03/2023,Games,749999.0
9,10,9,Henrique Vieira,808.808.808-08,Porto Alegre,RS,NaN,2023-03-21,Livros,330.0


---
## Bloco 1 — Diagnóstico geral

### Exercício 1
Verifique os **tipos de dado** de cada coluna e quantas informações **não-nulas** cada
uma tem.


In [ ]:
# Seu código aqui
df_vendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  20 non-null     int64  
 1   id_cliente          20 non-null     int64  
 2   nome                20 non-null     object 
 3   cpf                 20 non-null     object 
 4   cidade              18 non-null     object 
 5   estado              20 non-null     object 
 6   idade               17 non-null     float64
 7   data_cadastro       20 non-null     object 
 8   categoria_favorita  20 non-null     object 
 9   valor_gasto         18 non-null     float64
dtypes: float64(2), int64(2), object(6)
memory usage: 1.7+ KB


### Exercício 2
Gere um resumo estatístico de **todas** as colunas do dataset, incluindo as
categóricas/texto (não só as numéricas).


In [ ]:
# Seu código aqui
df_vendas.describe(include='all')

,id,id_cliente,nome,cpf,cidade,estado,idade,data_cadastro,categoria_favorita,valor_gasto
count,20.00000,20.00000,20,20,18,20,17.000000,20,20,18.000000
unique,NaN,NaN,18,18,11,14,NaN,18,3,NaN
top,NaN,NaN,Otávio Ramos,202.202.202-02,curitiba,pr,NaN,11/03/2023,Games,NaN
freq,NaN,NaN,3,3,4,3,NaN,3,8,NaN
mean,10.50000,9.75000,NaN,NaN,NaN,NaN,42.647059,NaN,NaN,42082.750000
std,5.91608,5.59017,NaN,NaN,NaN,NaN,44.508344,NaN,NaN,176672.844799
min,1.00000,1.00000,NaN,NaN,NaN,NaN,-3.000000,NaN,NaN,75.500000
25%,5.75000,5.00000,NaN,NaN,NaN,NaN,29.000000,NaN,NaN,226.250000
50%,10.50000,9.50000,NaN,NaN,NaN,NaN,33.000000,NaN,NaN,335.000000
75%,15.25000,14.25000,NaN,NaN,NaN,NaN,39.000000,NaN,NaN,822.500000


### Exercício 3
Descubra quantos valores nulos existem em cada coluna, tanto em **número absoluto**
quanto em **percentual** (arredondado para 1 casa decimal).


In [ ]:
# Seu código aqui
qtd_nulos = df_vendas.isnull().sum()
qtd_nulos

,0
id,0
id_cliente,0
nome,0
cpf,0
cidade,2
estado,0
idade,3
data_cadastro,0
categoria_favorita,0
valor_gasto,2


In [ ]:
prc_nulos = (df_vendas.isnull().mean() * 100).round(1)
prc_nulos

,0
id,0.0
id_cliente,0.0
nome,0.0
cpf,0.0
cidade,10.0
estado,0.0
idade,15.0
data_cadastro,0.0
categoria_favorita,0.0
valor_gasto,10.0


### Exercício 4
Use `nunique()` para ver quantos valores distintos existem na coluna `estado`. O número
bate com a quantidade de estados que realmente aparecem na base? O que isso sugere?


In [ ]:
# Seu código aqui
df_vendas['estado'].nunique()

14

In [ ]:
df_vendas['estado'].value_counts()

,count
estado,
pr,3
PR,2
PE,2
RS,2
CE,2
MG,1
sp,1
SP,1
BA,1


---
## Bloco 2 — Dados ausentes

### Exercício 5
A coluna `cidade` tem um valor `"-"` que é, na verdade, um nulo disfarçado. Substitua
esse valor por `NaN` de verdade (dica: `.replace()`).


In [ ]:
# Seu código aqui
df_vendas['cidade_replaced'] = df_vendas['cidade'].replace('-', np.nan)
df_vendas[['cidade', 'cidade_replaced']]

,cidade,cidade_replaced
0,Curitiba,Curitiba
1,curitiba,curitiba
2,Belo Horizonte,Belo Horizonte
3,None,None
4,curitiba,curitiba
5,curitiba,curitiba
6,Recife,Recife
7,-,NaN
8,Salvador,Salvador
9,Porto Alegre,Porto Alegre


### Exercício 6
Preencha os valores ausentes da coluna `idade` usando a **mediana**. Antes de fazer
isso, pense: por que a mediana costuma ser uma escolha mais segura que a média aqui?
(responda em uma célula de markdown ou em comentário)


In [ ]:
# Seu código aqui
df_vendas['idade_mediana'] = df_vendas['idade'].fillna(df_vendas['idade'].median())

df_vendas[['idade', 'idade_mediana']]

,idade,idade_mediana
0,29.0,29.0
1,33.0,33.0
2,NaN,33.0
3,41.0,41.0
4,33.0,33.0
5,33.0,33.0
6,37.0,37.0
7,210.0,210.0
8,26.0,26.0
9,NaN,33.0


In [ ]:
# O que fazer com o -3 e 210??


### Exercício 7
Preencha os valores ausentes da coluna `categoria_favorita` com a **moda** (valor mais
frequente).


In [ ]:
# Seu código aqui
df_vendas['categoria_favorita_moda'] = df_vendas['categoria_favorita'].fillna(df_vendas['categoria_favorita'].mode()[0])

df_vendas[['categoria_favorita', 'categoria_favorita_moda']]

,categoria_favorita,categoria_favorita_moda
0,Livros,Livros
1,Games,Games
2,Beleza,Beleza
3,Games,Games
4,Games,Games
5,Games,Games
6,Livros,Livros
7,Beleza,Beleza
8,Games,Games
9,Livros,Livros


### Exercício 8
Preencha os valores ausentes da coluna `cidade` com o texto `"Não Informado"`.


In [ ]:
# Seu código aqui
# df_vendas['cidade_replaced'] = df_vendas['cidade'].replace('-', np.nan)
df_vendas['cidade_na'] = df_vendas['cidade_replaced'].fillna('Não Informado')

df_vendas[['cidade', 'cidade_replaced', 'cidade_na']]

,cidade,cidade_replaced,cidade_na
0,Curitiba,Curitiba,Curitiba
1,curitiba,curitiba,curitiba
2,Belo Horizonte,Belo Horizonte,Belo Horizonte
3,None,None,Não Informado
4,curitiba,curitiba,curitiba
5,curitiba,curitiba,curitiba
6,Recife,Recife,Recife
7,-,NaN,Não Informado
8,Salvador,Salvador,Salvador
9,Porto Alegre,Porto Alegre,Porto Alegre


### Exercício 9
Em vez de usar a mediana geral da base inteira, calcule a **idade mediana por estado** e
use esse valor para preencher os nulos de `idade` (imputação por grupo, com
`groupby().transform()`).


In [ ]:
# Seu código aqui
df_vendas['idade_mediana_estado'] = df_vendas.groupby('estado')['idade'].transform(
    (lambda x: x.fillna(x.median())))

df_vendas[['estado', 'idade', 'idade_mediana_estado']]

,estado,idade,idade_mediana_estado
0,PR,29.0,29.0
1,pr,33.0,33.0
2,MG,NaN,NaN
3,SP,41.0,41.0
4,pr,33.0,33.0
5,pr,33.0,33.0
6,PE,37.0,37.0
7,sp,210.0,210.0
8,BA,26.0,26.0
9,RS,NaN,48.0


### Exercício 10 (bônus)
Antes de imputar `valor_gasto`, crie uma coluna booleana `valor_gasto_era_nulo` que
marca `True` onde o valor era originalmente ausente. Depois, preencha `valor_gasto` com
a mediana.


In [ ]:
# Seu código aqui
df_vendas['valor_gasto_era_nulo'] = df_vendas['valor_gasto'].isnull()

df_vendas['valor_gasto_era_nulo']

# df_vendas['valor_gasto'].isnull()

,valor_gasto_era_nulo
0,False
1,False
2,False
3,True
4,False
5,False
6,False
7,False
8,False
9,False


---
## Bloco 3 — Duplicatas

### Exercício 11
Quantas linhas **duplicadas exatas** (idênticas em todas as colunas) existem na base?


In [ ]:
# Seu código aqui
df_vendas.duplicated().sum()

### Exercício 12
Mostre **todas** as linhas envolvidas nessa duplicata (a linha "original" e a(s)
cópia(s)), não só as que o pandas marca como extras.


In [ ]:
# Seu código aqui
df_vendas[df_vendas.duplicated()]

### Exercício 13
Existe algum cliente com o mesmo `cpf` aparecendo mais de uma vez? Mostre essas linhas
usando `subset=['cpf']`.


In [ ]:
# Seu código aqui
df_vendas.duplicated(subset='cpf').sum()

In [ ]:
df_vendas[df_vendas.duplicated(subset='cpf')]

### Exercício 14
Remova as duplicatas da base, mantendo apenas **uma ocorrência por CPF** (não precisa se
preocupar com data desta vez — pode usar o padrão `keep='first'`).


In [ ]:
# Seu código aqui
df_vendas = df_vendas.drop_duplicates(subset='cpf', keep='first')
df_vendas.count()

---
## Bloco 4 — Outliers

### Exercício 15
Plote um **boxplot** da coluna `valor_gasto` para visualizar outliers.


In [ ]:
# Seu código aqui
df_vendas.boxplot(column='valor_gasto')

### Exercício 16
Calcule os limites inferior e superior pelo método **IQR** e identifique quais linhas da
base são outliers em `valor_gasto`.


Quando você divide os dados em 4 partes iguais (quartis), o IQR é a distância entre o quartil 1 (Q1) e o quartil 3 (Q3), ou seja, o intervalo onde ficam os 50% centrais dos dados, ignorando os 25% mais baixos e os 25% mais altos.


---

Fórmula:

IQR = Q3 - Q1


---


**Q1 (25º percentil):** 25% dos dados estão abaixo desse valor

**Q3 (75º percentil):** 75% dos dados estão abaixo desse valor

IQR: a "largura" da metade central da distribuição


---



Imagina o valor gasto de 8 clientes, já ordenado:

100, 150, 200, 250, 300, 350, 400, 9000

Q1 (25%) ≈ 187,5

Q3 (75%) ≈ 362,5

IQR = 362,5 − 187,5 = 175


---


O uso mais comum do IQR é achar valores atípicos, com uma regra bem estabelecida:


Limite inferior = Q1 - 1,5 × IQR

Limite superior = Q3 + 1,5 × IQR

---

Qualquer valor fora desse intervalo é considerado outlier. No exemplo acima:

Limite superior = 362,5 + 1,5 × 175 = 625

O valor 9000 está muito além disso → outlier

![https://www.scribbr.com/wp-content/uploads/2022/01/interquartile-range.png](https://www.scribbr.com/wp-content/uploads/2022/01/interquartile-range.png)

In [ ]:
# Seu código aqui
Q1 = df_vendas["valor_gasto"].quantile(0.25)
Q3 = df_vendas["valor_gasto"].quantile(0.75)
IQR = Q3 - Q1

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Limite inferior: {limite_inferior}")
print(f"Limite superior: {limite_superior}")

# Filtrando os outliers
outliers = df_vendas[(df_vendas["valor_gasto"] < limite_inferior) | (df_vendas["valor_gasto"] > limite_superior)]

outliers

### Exercício 18
Escolha **um** dos outliers de `valor_gasto` e decida (com justificativa) o que fazer
com ele: investigar, remover, manter e sinalizar, ou aplicar winsorização. Depois,
aplique a técnica escolhida com código.

*Dica: repare também na coluna `idade` — há valores ali que também parecem outliers
(ou erros de digitação). Vale aplicar o mesmo raciocínio.*


In [ ]:
# Seu código aqui

Q1 = df_vendas["idade"].quantile(0.25)
Q3 = df_vendas["idade"].quantile(0.75)
IQR = Q3 - Q1

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Limite inferior: {limite_inferior}")
print(f"Limite superior: {limite_superior}")

# print(df_vendas['idade'].std())

outliers = df_vendas[(df_vendas["idade"] < limite_inferior) | (df_vendas["idade"] > limite_superior)]
outliers

In [ ]:
# capping - limite inf como minima, limite sup como idade max

# mediana
df_vendas.at[7, "idade"] = np.random.randint(limite_inferior, limite_superior)

mediana = df_vendas["idade"].median()
print(mediana)
# df_vendas.head(10)

_Escreva sua resposta aqui._